# RAG

#### Odgovarjanje na vprašanja o nestrukturiranih podatkih
Cilj je razvoj agenta, ki lahko odgovarja na vprašanja o NBA pravilih.

In [ ]:
%%capture
!pip install google-genai unstructured psycopg2-binary pgvector beautifulsoup4

In [ ]:
nba_rules_urls = [
    "https://official.nba.com/rule-no-1-court-dimensions-equipment/",
    "https://official.nba.com/rule-no-2-duties-of-the-officials/",
    "https://official.nba.com/rule-no-3-players-substitutes-and-coaches/",
    "https://official.nba.com/rule-no-4-definitions/",
    "https://official.nba.com/rule-no-5-scoring-and-timing/",
    "https://official.nba.com/rule-no-6-putting-ball-in-play-live-dead-ball/",
    "https://official.nba.com/rule-no-7-24-second-clock/",
    "https://official.nba.com/rule-no-8-out-of-bounds-and-throw-in/",
    "https://official.nba.com/rule-no-9-free-throws-and-penalties/",
    "https://official.nba.com/rule-no-10-violations-and-penalties/",
    "https://official.nba.com/rule-no-11-basket-interference-goaltending/",
    "https://official.nba.com/rule-no-12-fouls-and-penalties/",
    "https://official.nba.com/rule-no-13-instant-replay/",
    "https://official.nba.com/rule-no-14-coachs-challenge/",
]

#### Nalaganje dokumentov

In [ ]:
import requests
from bs4 import BeautifulSoup

docs = []

for url in nba_rules_urls:
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        elements = soup.find_all(class_="col-xs-12 col-md-9")
        if len(elements) > 0:
            docs.append(elements[0].get_text(strip=True))
    except Exception as e:
        print(f"Error fetching {url}: {e}")

In [ ]:
len(docs)

#### Chunking

In [ ]:
from unstructured.chunking.basic import chunk_elements
from unstructured.documents.elements import Text

In [ ]:
docs = [Text(doc) for doc in docs]
chunks = chunk_elements(docs, max_characters=1000, overlap=200)

In [ ]:
len(chunks)

#### Embeddings

In [ ]:
import os
from google import genai

In [ ]:
os.environ["GOOGLE_API_KEY"] = ""

In [ ]:
texts = [chunk.text for chunk in chunks]

In [ ]:
client = genai.Client()

embeddings = []
batch_size = 50
for i in range(0, len(texts), batch_size):
    batch = texts[i : i + batch_size]
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=batch
    )
    embeddings.extend([emb.values for emb in response.embeddings])

In [ ]:
len(embeddings)

#### Vektorska baza

In [ ]:
from sqlalchemy import create_engine, Column, Integer, Text
from sqlalchemy.orm import declarative_base, sessionmaker
from pgvector.sqlalchemy import Vector

Base = declarative_base()


class Document(Base):
    __tablename__ = "documents"

    id = Column(Integer, primary_key=True)
    text = Column(Text)
    embedding = Column(Vector(3072))

In [ ]:
pg_url = ""
pg_user = ""
pg_password = ""

In [ ]:
engine = create_engine(
    f"postgresql+psycopg2://{pg_user}:{pg_password}@{pg_url}/postgres"
)

# Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session = Session()

In [ ]:
# vstavljanje v bazo
# docs = [Document(text=t, embedding=e) for t, e in zip(texts, embeddings)]
# session.add_all(docs)
# session.commit()

In [ ]:
def get_embedding(text: str) -> list[float]:
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[text]
    )
    return response.embeddings[0].values

In [ ]:
def find_similar_documents(query: str, top_k: int = 5) -> list[str]:
    query_embedding = get_embedding(query)

    results = (
        session.query(Document)
        .order_by(Document.embedding.cosine_distance(query_embedding))
        .limit(top_k)
        .all()
    )

    return [result.text for result in results]

In [ ]:
find_similar_documents("What is a double dribble?")